# 3. Spreading work over banks

More banks means more tiles computed at once. Here the same kernel is compiled
for 8, 16, 32 and 64 banks. DDR4 has 32 banks per channel, so 64 banks is the
first configuration that spans **two channels**, two memory controllers working
in parallel.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # tutorial/, for _infra
from _infra import pud

NAME, MEM = "vadd_1048576", "ddr"
BANKS = [8, 16, 32, 64]

rows = []
for banks in BANKS:
    k = pud.compile_kernel(NAME, banks, MEM)
    loop = pud.mapping_attrs(k.mapping.path)[0]
    s = pud.simulate(k.exe, MEM)
    rows.append(dict(banks=banks, parallel=loop["parallel_factor"],
                     per_bank=loop["iters_per_bank"],
                     channels=-(-banks // pud.MEMORIES[MEM][1]),
                     row_ops=s.row_ops, us=s.makespan_us,
                     cyc_per_op=s.extra["cycles"] / s.row_ops))
    print(f"{banks:3} banks done ({s.seconds:.0f} s)")

In [ ]:
print(f"{'banks':>5} {'channels':>8} {'parallel':>8} {'tiles/bank':>10} "
      f"{'row ops':>8} {'makespan us':>11} {'speed-up':>8} {'core cyc/op':>11}")
for r in rows:
    print(f"{r['banks']:5} {r['channels']:8} {r['parallel']:8} {r['per_bank']:10} "
          f"{r['row_ops']:8} {r['us']:11.3f} {rows[0]['us'] / r['us']:8.2f} {r['cyc_per_op']:11.2f}")

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot([r["banks"] for r in rows], [r["us"] for r in rows], "o-", label="compiled")
ax.plot([r["banks"] for r in rows],
        [rows[0]["us"] * rows[0]["banks"] / r["banks"] for r in rows], "--",
        color="gray", label="perfect scaling")
ax.axvspan(32.5, 70, color="0.93", label="two channels")
ax.set_xscale("log", base=2); ax.set_xticks(BANKS); ax.set_xticklabels(BANKS)
ax.set_xlabel("banks"); ax.set_ylabel("makespan (us)"); ax.set_title(NAME)
ax.legend(); plt.tight_layout()

**What to look for.**

- The number of commands does not change: the work is the same, only its
  distribution.
- Up to 32 banks, doubling the banks roughly halves the makespan: every command
  issued to a bank loop reaches more banks at once.
- At 64 banks the memory has twice the controllers, but the **core** still issues
  every command, one instruction at a time. The core's cycles per command are
  now what limits the speed-up. Notebook 04 separates the two effects.

> **Try it:** set `MEM = "hbm"` (16 banks per channel, so the second channel
> starts at 32 banks), or sweep `NAME = "popcount_1048576"`.